# Case Study 3: Automotive Cybersecurity TARA Assistant
**Automotive Engineering AI Platform | Priority: Phase 2 | Value: improved threat coverage and reusable mitigation knowledge**

Run top to bottom (Runtime -> Run all). GPU is only needed if you switch on LLM narratives.

| Common design principle | Where it is implemented |
|---|---|
| Open-source LLM | `Qwen2.5-1.5B-Instruct` (optional, off by default), rule-based fallback |
| Data stays inside org boundary | Everything runs locally / no external API calls |
| Retrieval-Augmented Generation | Sections 4 + 6: threats and controls come only from retrieved KB documents |
| Local vector store | FAISS (per-project index, saved to disk) |
| FastAPI service layer | Section 7 |
| Simple engineering web UI | Section 9 (Gradio) |
| Citations, confidence, audit log, versioning, human approval | Sections 5, 6, 7, 8 |
| Separate project collections + RBAC | Sections 3, 4, 5 |

**Fixes in this version:** UI now refreshes every tab when the signed-in user changes (no stale "403" results), the Audit tab auto-loads when opened, API returns proper 404/400 instead of 500, query embeddings are cached, LLM narratives are batched, and the evaluation sweep reuses cached searches.

> Illustrative sample knowledge only. The assistant supports, and does not replace, the security engineer's decision.

## 1. Install

In [ ]:
!pip -q install fastapi uvicorn faiss-cpu sentence-transformers transformers accelerate gradio nest_asyncio pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 70.5 MB/s eta 0:00:00


## 2. Config and models

In [ ]:
import os, json, time, hashlib, uuid, threading, re, datetime as dt
from functools import lru_cache
import numpy as np, pandas as pd, requests, faiss
from typing import Optional, List
from sentence_transformers import SentenceTransformer

BASE = "/content/tara_platform"
os.makedirs(BASE, exist_ok=True)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_NAME    = "Qwen/Qwen2.5-1.5B-Instruct"   # any approved open-source instruct model
USE_LLM     = False                          # False = fast template mode (can be switched on from the UI)
SHARE_LINK  = True                           # False = skip the public gradio.live link (slightly faster start)

embedder = SentenceTransformer(EMBED_MODEL)

@lru_cache(maxsize=512)
def embed_query(text):
    """Cached query embedding: repeated searches (evaluation sweeps, re-generation) cost nothing."""
    return embedder.encode([text], normalize_embeddings=True).astype("float32")

llm = None
if USE_LLM:
    try:
        from transformers import pipeline
        llm = pipeline("text-generation", model=LLM_NAME, torch_dtype="auto", device_map="auto")
        print("LLM loaded:", LLM_NAME)
    except Exception as e:
        print("LLM not loaded, using template fallback ->", e)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 3. Users, roles and project access (RBAC)

In [ ]:
USERS = {
    "analyst-key":     {"user": "alice", "role": "analyst",       "projects": ["project_adas"]},
    "infotainment-key":{"user": "carol", "role": "analyst",       "projects": ["project_infotainment"]},
    "lead-key":        {"user": "bob",   "role": "security_lead", "projects": ["project_adas", "project_infotainment"]},
    "auditor-key":     {"user": "dave",  "role": "auditor",       "projects": ["project_adas", "project_infotainment"]},
}
PERMS = {
    "analyst":       {"ingest", "generate", "read", "revise"},
    "security_lead": {"ingest", "generate", "read", "revise", "approve", "audit"},
    "auditor":       {"read", "audit"},
}
def authorize(u, project, action):
    if action not in PERMS[u["role"]]:
        raise PermissionError(f"Role '{u['role']}' cannot '{action}'")
    if project and project not in u["projects"]:
        raise PermissionError(f"User '{u['user']}' has no access to '{project}'")

## 4. Knowledge base and local FAISS vector store (one collection per project)

In [ ]:
SRC = "Internal TARA Knowledge Base v1.0 (illustrative)"
# ap = (elapsed time, expertise, knowledge, window of opportunity, equipment) attack-potential points
def T(id, title, stride, asset_types, text, S, F, O, P, ap, controls):
    return dict(id=id, type="threat", title=title, source=SRC, stride=stride, asset_types=asset_types, text=text,
                impact=dict(Safety=S, Financial=F, Operational=O, Privacy=P), ap=ap, controls=controls)

KB = {
 "project_adas": [
  T("THR-ADAS-001","CAN message spoofing / injection","Spoofing",["ECU","CAN bus","gateway"],
    "An attacker with access to the in-vehicle CAN network injects forged frames impersonating a trusted ECU, such as brake or steering commands, because classic CAN has no sender authentication.",
    3,2,2,0,(1,3,3,4,0),
    ["SecOC message authentication with freshness values","Gateway filtering and network segmentation","Intrusion detection on CAN anomalies"]),
  T("THR-ADAS-002","Malicious or tampered OTA software update","Tampering",["OTA","telematics","ECU","bootloader"],
    "An attacker manipulates an over-the-air update package or the update channel so that modified firmware is installed on an ADAS controller.",
    3,3,3,1,(4,6,3,4,4),
    ["Signed update packages with chain of trust","Secure boot with rollback protection","Mutual TLS between vehicle and backend"]),
  T("THR-ADAS-003","Diagnostic interface abuse (UDS / OBD-II)","Elevation of privilege",["diagnostic","OBD","UDS","ECU"],
    "An attacker connected to the OBD-II port abuses UDS services such as security access, routine control or reflashing to gain privileged control of an ECU.",
    2,2,2,0,(4,3,3,4,0),
    ["Strong challenge-response security access with unique keys","Session and service-level access control","Diagnostic gateway with authentication and logging"]),
  T("THR-ADAS-004","Sensor spoofing (GNSS, camera, radar)","Spoofing",["sensor","GNSS","camera","radar","ADAS"],
    "An attacker transmits spoofed GNSS signals or adversarial inputs to perception sensors causing wrong position or object detection and unsafe driving decisions.",
    3,1,2,0,(4,6,3,4,4),
    ["Sensor plausibility and cross-validation","Multi-constellation GNSS with authentication","Fail-operational degradation strategy"]),
  T("THR-ADAS-005","Extraction of cryptographic keys from ECU","Information disclosure",["ECU","HSM","key storage","bootloader"],
    "An attacker with physical access uses debug ports or fault injection to read cryptographic keys from flash, enabling cloning and bypass of authentication.",
    1,3,2,1,(17,6,7,10,7),
    ["Hardware security module for key storage","Disable or lock debug interfaces in production","Unique per-vehicle keys"]),
  T("THR-ADAS-006","Remote exploitation via telematics unit","Elevation of privilege",["telematics","TCU","cellular","gateway"],
    "An attacker exploits a vulnerability in the cellular telematics unit, pivots through the gateway and sends commands to safety-relevant domains.",
    3,3,3,2,(4,6,3,1,0),
    ["Hardened OS and minimal exposed services","Firewall between telematics and vehicle networks","Vulnerability monitoring and patch process"]),
  T("THR-ADAS-007","Denial of service by bus flooding","Denial of service",["CAN bus","ECU","gateway","Ethernet"],
    "An attacker floods the network with high priority messages, delaying or blocking safety-critical communication between ADAS controllers.",
    2,1,2,0,(1,0,0,4,0),
    ["Rate limiting at the gateway","Bus-off monitoring and recovery","Message priority and bandwidth allocation"]),
  T("THR-ADAS-008","Debug interface (JTAG/SWD) left active","Tampering",["ECU","debug","bootloader"],
    "An attacker connects to an enabled debug port and modifies memory or firmware to change ADAS behavior.",
    2,2,1,0,(4,6,3,10,4),
    ["Permanently disable debug in production","Authenticated debug unlock","Secure boot verification"]),
  dict(id="REF-ADAS-001", type="reference", title="TARA process guidance", source=SRC, stride="",
       text="TARA steps: identify assets and security properties, derive damage scenarios, identify threat scenarios, analyse attack paths, rate attack feasibility, determine risk value and select risk treatment. Impact categories are Safety, Financial, Operational and Privacy."),
 ],
 "project_infotainment": [
  T("THR-IVI-001","Malicious USB device attack","Tampering",["infotainment","USB","IVI"],
    "An attacker inserts a crafted USB device that exploits file system or driver parsing flaws on the head unit to execute code.",
    0,2,2,1,(1,3,3,1,0),
    ["USB device class whitelisting","Sandboxed media parsers","Disable auto-run and unnecessary drivers"]),
  T("THR-IVI-002","Bluetooth pairing attack","Spoofing",["infotainment","Bluetooth","IVI"],
    "A nearby attacker abuses weak pairing or a protocol vulnerability to access contacts, messages or to execute code on the head unit.",
    0,1,1,3,(1,6,3,4,0),
    ["Secure Simple Pairing with user confirmation","Timely Bluetooth stack patching","Limit exposed profiles"]),
  T("THR-IVI-003","Malicious third-party app","Elevation of privilege",["infotainment","app store","IVI"],
    "A malicious application obtains excessive permissions and leaks personal data or interferes with vehicle functions through the head unit.",
    1,2,2,3,(4,3,0,1,0),
    ["App signing and vetting","Least-privilege permission model","Isolation between IVI and vehicle domains"]),
 ],
}

class VectorStore:
    def __init__(self): self.idx, self.meta = {}, {}
    def add(self, collection, docs, persist=True):
        texts = [d["title"] + ". " + d["text"] for d in docs]
        emb = embedder.encode(texts, normalize_embeddings=True, batch_size=64).astype("float32")
        if collection not in self.idx:
            self.idx[collection] = faiss.IndexFlatIP(emb.shape[1]); self.meta[collection] = []
        self.idx[collection].add(emb); self.meta[collection].extend(docs)
        if persist: self.save(collection)
    def search(self, collection, query, k=8):
        if collection not in self.idx: return []
        q = embed_query(query)
        sc, ix = self.idx[collection].search(q, min(k, len(self.meta[collection])))
        return [(self.meta[collection][i], float(s)) for s, i in zip(sc[0], ix[0]) if i >= 0]
    def save(self, collection):
        d = f"{BASE}/vectorstore"; os.makedirs(d, exist_ok=True)
        faiss.write_index(self.idx[collection], f"{d}/{collection}.faiss")
        with open(f"{d}/{collection}.json", "w") as f: json.dump(self.meta[collection], f)

store = VectorStore()
for coll, docs in KB.items():
    store.add(coll, docs)
    print(coll, "->", len(docs), "documents indexed")

project_adas -> 9 documents indexed
project_infotainment -> 3 documents indexed


## 5. Audit log (tamper-evident hash chain) and versioned TARA records

In [ ]:
AUDIT_FILE = f"{BASE}/audit.jsonl"
open(AUDIT_FILE, "w").close()
_audit_lock = threading.Lock()
_last_hash = ["GENESIS"]
_now = lambda: dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")

def audit(user, action, project, details):
    with _audit_lock:
        e = dict(ts=_now(), user=user, action=action, project=project, details=details, prev=_last_hash[0])
        e["hash"] = hashlib.sha256(json.dumps(e, sort_keys=True).encode()).hexdigest()
        _last_hash[0] = e["hash"]
        with open(AUDIT_FILE, "a") as f: f.write(json.dumps(e) + "\n")

def read_audit():
    with open(AUDIT_FILE) as f: return [json.loads(l) for l in f if l.strip()]

def verify_audit():
    prev = "GENESIS"
    for e in read_audit():
        h = e.pop("hash")
        if e["prev"] != prev or hashlib.sha256(json.dumps(e, sort_keys=True).encode()).hexdigest() != h: return False
        prev = h
    return True

RECORDS = {}   # tara_id -> list of versions
def save_records():
    with open(f"{BASE}/tara_records.json", "w") as f: json.dump(RECORDS, f, indent=1)

## 6. TARA engine (ISO/SAE 21434 style: impact, attack feasibility, risk, treatment) with RAG

In [ ]:
IMPACT_NAMES = ["Negligible", "Moderate", "Major", "Severe"]
FEAS_IDX = {"Very Low": 0, "Low": 1, "Medium": 2, "High": 3}
MATRIX = {0: [1,1,1,1], 1: [1,2,2,3], 2: [1,2,3,4], 3: [1,3,4,5]}   # impact x feasibility -> risk 1..5 (configurable)

def feasibility(points):
    s = sum(points)
    return s, ("High" if s <= 13 else "Medium" if s <= 19 else "Low" if s <= 24 else "Very Low")
def risk_value(impact_level, feas): return MATRIX[impact_level][FEAS_IDX[feas]]
def treatment(r):
    return {5:"Reduce (mandatory)",4:"Reduce (mandatory)",3:"Reduce (recommended)",2:"Retain or Share (justify)",1:"Retain"}[r]
def confidence(score):
    return "High" if score >= 0.50 else "Medium" if score >= 0.35 else "Low"

def _fallback_narr(asset, meta):
    return (f"{meta['title']} against {asset['name']}: {meta['text']}",
            f"Loss of {', '.join(k for k,v in meta['impact'].items() if v >= 2) or 'integrity'} for {asset['name']}.")

def _prompt(asset, meta):
    return (f"You are an automotive cybersecurity engineer. Using ONLY this context, write two short lines.\n"
            f"Context: {meta['text']}\nAsset: {asset['name']} - {asset.get('description','')}\n"
            f"Line 1 starts with 'THREAT:' (threat scenario for this asset). Line 2 starts with 'DAMAGE:' (damage scenario).")

_narr_cache = {}
def narrate_many(asset, metas):
    """One batched LLM call for all threats (much faster than one call each); cached; template fallback."""
    fb = [_fallback_narr(asset, m) for m in metas]
    if llm is None or not metas: return fb
    keys = [(asset["name"], asset.get("description", ""), m["id"]) for m in metas]
    todo = [i for i, k in enumerate(keys) if k not in _narr_cache]
    if todo:
        try:
            chats = [[{"role": "user", "content": _prompt(asset, metas[i])}] for i in todo]
            outs = llm(chats, max_new_tokens=80, do_sample=False, batch_size=len(chats))
            for i, o in zip(todo, outs):
                o = o[0] if isinstance(o, list) else o
                txt = o["generated_text"][-1]["content"].strip()
                t = re.search(r"THREAT:\s*(.*)", txt); d = re.search(r"DAMAGE:\s*(.*)", txt)
                _narr_cache[keys[i]] = ((t.group(1).strip() if t else fb[i][0]), (d.group(1).strip() if d else fb[i][1]))
        except Exception as e:
            print("LLM batch failed, using templates ->", e)
    return [_narr_cache.get(k, f) for k, f in zip(keys, fb)]

def build_row(asset, meta, score, ap=None, note="", narr=None):
    ap = ap or meta["ap"]
    lvl = max(meta["impact"].values())
    pts, feas = feasibility(ap)
    r = risk_value(lvl, feas)
    ts, ds = narr or _fallback_narr(asset, meta)
    c = confidence(score)
    return dict(threat_id=meta["id"], stride=meta["stride"], threat_scenario=ts, damage_scenario=ds,
                impact=meta["impact"], impact_rating=IMPACT_NAMES[lvl], attack_potential=list(ap), attack_potential_sum=pts,
                feasibility=feas, risk_value=r, treatment=treatment(r), controls=meta["controls"],
                citations=[dict(doc_id=meta["id"], title=meta["title"], source=meta["source"], similarity=round(score, 3))],
                confidence=c, needs_expert_review=(c == "Low"), analyst_note=note)

def generate_tara(u, project, asset):
    authorize(u, project, "generate")
    query = f"{asset['asset_type']} {asset['name']} {asset['description']} {asset.get('interfaces','')}"
    hits = [(m, s) for m, s in store.search(project, query, k=8) if m["type"] == "threat" and s >= 0.25][:6]
    narrs = narrate_many(asset, [m for m, _ in hits])
    rows = [build_row(asset, m, s, narr=n) for (m, s), n in zip(hits, narrs)]
    rows.sort(key=lambda r: -r["risk_value"])
    tid = "TARA-" + uuid.uuid4().hex[:8].upper()
    rec = dict(tara_id=tid, version=1, project=project, asset=asset, rows=rows, status="DRAFT (human approval required)",
               created_by=u["user"], created_at=_now(), approved_by=None,
               warning=None if rows else "No grounded threats retrieved. Escalate to a security engineer.")
    RECORDS[tid] = [rec]; save_records()
    audit(u["user"], "generate", project, dict(tara_id=tid, asset=asset["name"], threats=len(rows)))
    return rec

def revise_tara(u, tid, row_index, ap, note):
    v = RECORDS[tid][-1]; authorize(u, v["project"], "revise")
    if not (0 <= row_index < len(v["rows"])): raise ValueError(f"row_index must be 0..{len(v['rows'])-1}")
    if len(ap) != 5: raise ValueError("attack_potential needs 5 values")
    new = json.loads(json.dumps(v))
    row = new["rows"][row_index]
    meta = next(m for m in store.meta[v["project"]] if m["id"] == row["threat_id"])
    upd = build_row(v["asset"], meta, row["citations"][0]["similarity"], ap=ap, note=f"Revised by {u['user']}: {note}",
                    narr=(row["threat_scenario"], row["damage_scenario"]))   # reuse text: no LLM call needed
    new["rows"][row_index] = upd
    new["version"] = v["version"] + 1; new["status"] = "DRAFT (human approval required)"; new["approved_by"] = None
    RECORDS[tid].append(new); save_records()
    audit(u["user"], "revise", v["project"], dict(tara_id=tid, version=new["version"], row=row_index, note=note))
    return new

def approve_tara(u, tid):
    v = RECORDS[tid][-1]; authorize(u, v["project"], "approve")
    v["status"] = "APPROVED"; v["approved_by"] = u["user"]; save_records()
    audit(u["user"], "approve", v["project"], dict(tara_id=tid, version=v["version"]))
    return v

def ingest_doc(u, project, title, text, source):
    authorize(u, project, "ingest")
    chunks = [c.strip() for c in re.split(r"\n\s*\n", text) if len(c.strip()) > 40] or [text]
    docs = [dict(id=f"REF-{uuid.uuid4().hex[:6].upper()}", type="reference", title=f"{title} (part {i+1})", source=source, stride="", text=c) for i, c in enumerate(chunks)]
    store.add(project, docs)
    audit(u["user"], "ingest", project, dict(title=title, chunks=len(docs)))
    return len(docs)

def rows_to_df(rec):
    return pd.DataFrame([{
        "Threat": r["threat_id"], "STRIDE": r["stride"], "Impact": r["impact_rating"], "AP sum": r["attack_potential_sum"],
        "Feasibility": r["feasibility"], "Risk": r["risk_value"], "Treatment": r["treatment"],
        "Controls": "; ".join(r["controls"]), "Confidence": r["confidence"], "Review": "YES" if r["needs_expert_review"] else "",
        "Citation": f"{r['citations'][0]['doc_id']} ({r['citations'][0]['similarity']})"} for r in rec["rows"]])

## 7. FastAPI service layer

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import nest_asyncio, uvicorn
nest_asyncio.apply()

app = FastAPI(title="Automotive Cybersecurity TARA Assistant")

class AssetIn(BaseModel):
    name: str
    description: str = ""
    asset_type: str = "ECU"
    interfaces: str = ""
class IngestIn(BaseModel):
    title: str
    text: str
    source: str = "User upload"
class ReviseIn(BaseModel):
    row_index: int
    attack_potential: List[int]
    note: str = ""

def user_of(key):
    if key not in USERS: raise HTTPException(401, "Invalid API key")
    return USERS[key]
def guard(fn, *a):
    try: return fn(*a)
    except PermissionError as e: raise HTTPException(403, str(e))
    except (ValueError, IndexError) as e: raise HTTPException(400, str(e))
def versions_of(tid):
    if tid not in RECORDS: raise HTTPException(404, f"TARA '{tid}' not found")
    return RECORDS[tid]

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/projects/{project}/ingest")
def api_ingest(project: str, body: IngestIn, x_api_key: str = Header(...)):
    return {"chunks": guard(ingest_doc, user_of(x_api_key), project, body.title, body.text, body.source)}

@app.post("/projects/{project}/tara")
def api_generate(project: str, asset: AssetIn, x_api_key: str = Header(...)):
    return guard(generate_tara, user_of(x_api_key), project, asset.dict())

@app.get("/tara/{tid}")
def api_get(tid: str, version: Optional[int] = None, x_api_key: str = Header(...)):
    u = user_of(x_api_key); vs = versions_of(tid)
    if version and not (1 <= version <= len(vs)): raise HTTPException(404, "Version not found")
    v = vs[(version - 1) if version else -1]
    guard(authorize, u, v["project"], "read")
    audit(u["user"], "read", v["project"], dict(tara_id=tid, version=v["version"]))
    return v

@app.get("/tara/{tid}/versions")
def api_versions(tid: str, x_api_key: str = Header(...)):
    u = user_of(x_api_key); vs = versions_of(tid); guard(authorize, u, vs[-1]["project"], "read")
    return [dict(version=v["version"], status=v["status"], approved_by=v["approved_by"]) for v in vs]

@app.post("/tara/{tid}/revise")
def api_revise(tid: str, body: ReviseIn, x_api_key: str = Header(...)):
    versions_of(tid)
    return guard(revise_tara, user_of(x_api_key), tid, body.row_index, body.attack_potential, body.note)

@app.post("/tara/{tid}/approve")
def api_approve(tid: str, x_api_key: str = Header(...)):
    versions_of(tid)
    return guard(approve_tara, user_of(x_api_key), tid)

@app.get("/audit")
def api_audit(x_api_key: str = Header(...)):
    u = user_of(x_api_key); guard(authorize, u, None, "audit")
    return dict(chain_valid=verify_audit(), entries=read_audit())

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=1).ok: print("API running on http://127.0.0.1:8000"); break
    except Exception: time.sleep(0.2)

API running on http://127.0.0.1:8000


## 8. End-to-end demo (generate -> cite -> revise -> RBAC checks -> approve -> audit)

In [ ]:
API = "http://127.0.0.1:8000"
H = lambda k: {"x-api-key": k}

asset = {"name": "ADAS Domain Controller", "asset_type": "ECU",
         "description": "Fuses camera and radar data, sends brake and steering requests over CAN, supports UDS reflashing and OTA updates",
         "interfaces": "CAN, Ethernet, OBD-II, telematics gateway"}

# 1) Analyst generates a draft TARA
rec = requests.post(f"{API}/projects/project_adas/tara", json=asset, headers=H("analyst-key")).json()
tid = rec["tara_id"]; print(tid, "| v", rec["version"], "|", rec["status"])
display(rows_to_df(rec))

TARA-22BFFAB8 | v 1 | DRAFT (human approval required)


/tmp/ipykernel_3664/945274079.py:42: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  return guard(generate_tara, user_of(x_api_key), project, asset.dict())


,Threat,STRIDE,Impact,AP sum,Feasibility,Risk,Treatment,Controls,Confidence,Review,Citation
0,THR-ADAS-001,Spoofing,Severe,11,High,5,Reduce (mandatory),SecOC message authentication with freshness va...,Medium,,THR-ADAS-001 (0.465)
1,THR-ADAS-007,Denial of service,Major,5,High,4,Reduce (mandatory),Rate limiting at the gateway; Bus-off monitori...,Medium,,THR-ADAS-007 (0.397)
2,THR-ADAS-006,Elevation of privilege,Severe,14,Medium,4,Reduce (mandatory),Hardened OS and minimal exposed services; Fire...,Low,YES,THR-ADAS-006 (0.349)
3,THR-ADAS-003,Elevation of privilege,Major,14,Medium,3,Reduce (recommended),Strong challenge-response security access with...,Medium,,THR-ADAS-003 (0.468)
4,THR-ADAS-002,Tampering,Severe,21,Low,3,Reduce (recommended),Signed update packages with chain of trust; Se...,Medium,,THR-ADAS-002 (0.423)
5,THR-ADAS-005,Information disclosure,Severe,47,Very Low,1,Retain,Hardware security module for key storage; Disa...,Medium,,THR-ADAS-005 (0.463)


In [ ]:
# 2) RBAC: analyst of another project is blocked; analyst cannot approve
print("Cross-project  :", requests.get(f"{API}/tara/{tid}", headers=H("infotainment-key")).status_code, "(expect 403)")
print("Analyst approve:", requests.post(f"{API}/tara/{tid}/approve", headers=H("analyst-key")).status_code, "(expect 403)")
print("Analyst audit  :", requests.get(f"{API}/audit", headers=H("analyst-key")).status_code, "(expect 403)")
print("Auditor audit  :", requests.get(f"{API}/audit", headers=H("auditor-key")).status_code, "(expect 200)")

# 3) Human revision (new version) - e.g. analyst judges the attack harder than the KB default
r2 = requests.post(f"{API}/tara/{tid}/revise", json={"row_index": 0, "attack_potential": [17, 6, 7, 4, 4], "note": "Needs bench access and specialised tools"}, headers=H("analyst-key")).json()
print("Revised -> version", r2["version"], "| row0 risk:", r2["rows"][0]["risk_value"])

# 4) Security lead approves
r3 = requests.post(f"{API}/tara/{tid}/approve", headers=H("lead-key")).json()
print("Status:", r3["status"], "by", r3["approved_by"])
print("Versions:", requests.get(f"{API}/tara/{tid}/versions", headers=H("lead-key")).json())

Cross-project  : 403 (expect 403)
Analyst approve: 403 (expect 403)
Analyst audit  : 403 (expect 403)
Auditor audit  : 200 (expect 200)
Revised -> version 2 | row0 risk: 1
Status: APPROVED by bob
Versions: [{'version': 1, 'status': 'DRAFT (human approval required)', 'approved_by': None}, {'version': 2, 'status': 'APPROVED', 'approved_by': 'bob'}]


In [ ]:
# 5) Ingest a project document, then check the audit trail (hash chain)
print(requests.post(f"{API}/projects/project_adas/ingest", headers=H("analyst-key"),
      json={"title": "ADAS security concept", "text": "The ADAS controller must authenticate all safety messages.\n\nOTA updates are only accepted if signed by the OEM backend and verified by the bootloader.", "source": "ADAS-SecConcept-v2"}).json())
a = requests.get(f"{API}/audit", headers=H("auditor-key")).json()
print("Audit chain valid:", a["chain_valid"])
display(pd.DataFrame(a["entries"])[["ts", "user", "action", "project", "details"]])

# Export approved TARA to CSV
rows_to_df(requests.get(f"{API}/tara/{tid}", headers=H("lead-key")).json()).to_csv(f"/content/{tid}.csv", index=False)
print("Saved", f"/content/{tid}.csv")

{'chunks': 2}
Audit chain valid: True


,ts,user,action,project,details
0,2026-10-06T17:39:52.457728Z,alice,generate,project_adas,"{'tara_id': 'TARA-22BFFAB8', 'asset': 'ADAS Do..."
1,2026-10-06T17:39:52.577598Z,alice,revise,project_adas,"{'tara_id': 'TARA-22BFFAB8', 'version': 2, 'ro..."
2,2026-10-06T17:39:52.584035Z,bob,approve,project_adas,"{'tara_id': 'TARA-22BFFAB8', 'version': 2}"
3,2026-10-06T17:39:52.671486Z,alice,ingest,project_adas,"{'title': 'ADAS security concept', 'chunks': 2}"


Saved /content/TARA-22BFFAB8.csv


## 9. Web interface (Gradio): Dashboard, New TARA, Review, Accuracy, Audit, Knowledge Base

In [ ]:
import html, inspect as _inspect, gradio as gr, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

PALETTE = {1: "#22c55e", 2: "#84cc16", 3: "#f59e0b", 4: "#f97316", 5: "#ef4444"}
RISK_LABEL = {1: "Very Low", 2: "Low", 3: "Medium", 4: "High", 5: "Critical"}
CONF_COL = {"High": "#16a34a", "Medium": "#d97706", "Low": "#dc2626"}
llm_backup = [llm]
PROJECTS = ["project_adas", "project_infotainment"]

# ---------------- Design system ----------------
plt.rcParams.update({
    "figure.facecolor": "#0d1424", "axes.facecolor": "#0d1424", "savefig.facecolor": "#0d1424",
    "axes.edgecolor": "#273449", "axes.labelcolor": "#94a3b8", "text.color": "#e2e8f0",
    "xtick.color": "#94a3b8", "ytick.color": "#94a3b8", "axes.titlecolor": "#f1f5f9",
    "axes.titlesize": 12, "axes.titleweight": "semibold", "axes.titlelocation": "left", "axes.titlepad": 12,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#1c2740", "grid.linewidth": .8, "axes.axisbelow": True,
    "legend.frameon": False, "legend.labelcolor": "#cbd5e1", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False})

THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.indigo, secondary_hue=gr.themes.colors.violet, neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "monospace"],
    radius_size=gr.themes.sizes.radius_lg,
).set(
    body_background_fill_dark="#070b14", background_fill_primary_dark="#0d1424", background_fill_secondary_dark="#0a1020",
    block_background_fill_dark="#0d1424", block_border_color_dark="#1e293b", block_border_width="1px", block_shadow="none",
    block_label_background_fill_dark="transparent", block_label_text_color_dark="#94a3b8", block_title_text_color_dark="#cbd5e1",
    input_background_fill_dark="#111a2e", input_border_color_dark="#243049", input_border_color_focus_dark="#6366f1",
    button_primary_background_fill="linear-gradient(135deg,#6366f1,#8b5cf6)",
    button_primary_background_fill_hover="linear-gradient(135deg,#7a7cf6,#9d78f8)",
    button_primary_text_color="white", button_primary_border_color="transparent",
    button_secondary_background_fill_dark="#111a2e", button_secondary_background_fill_hover_dark="#18233c",
    button_secondary_border_color_dark="#243049", button_secondary_text_color_dark="#e2e8f0",
    table_even_background_fill_dark="#0d1424", table_odd_background_fill_dark="#0a1020", table_border_color_dark="#1e293b",
    table_row_focus_dark="#18233c", slider_color="#6366f1", checkbox_background_color_selected="#6366f1")

FORCE_DARK = "() => { document.body.classList.add('dark'); }"

CSS = """
:root{--bg:#070b14;--surface:#0d1424;--surface2:#111a2e;--line:#1e293b;--line2:#2a3856;--tx:#e5e7eb;--mut:#94a3b8;--acc:#6366f1;--acc2:#8b5cf6}
body,.gradio-container{background:radial-gradient(1100px 480px at 12% -8%,rgba(99,102,241,.20),transparent 60%),radial-gradient(900px 420px at 100% 0%,rgba(34,211,238,.09),transparent 55%),var(--bg) !important}
.gradio-container{max-width:1320px !important;margin:0 auto !important}
footer{display:none !important}
::-webkit-scrollbar{width:10px;height:10px}::-webkit-scrollbar-thumb{background:#243049;border-radius:8px}
.hero{position:relative;overflow:hidden;display:flex;justify-content:space-between;align-items:center;gap:24px;flex-wrap:wrap;padding:28px 32px;border-radius:22px;
 border:1px solid rgba(129,140,248,.28);background:linear-gradient(135deg,rgba(30,58,138,.65),rgba(79,70,229,.50) 55%,rgba(124,58,237,.50));
 box-shadow:0 24px 60px -24px rgba(79,70,229,.65),inset 0 1px 0 rgba(255,255,255,.10)}
.hero::after{content:"";position:absolute;inset:0;pointer-events:none;background-image:linear-gradient(rgba(255,255,255,.045) 1px,transparent 1px),linear-gradient(90deg,rgba(255,255,255,.045) 1px,transparent 1px);background-size:34px 34px;-webkit-mask-image:linear-gradient(90deg,transparent,#000 75%);mask-image:linear-gradient(90deg,transparent,#000 75%)}
.hero-l,.hero-r{position:relative;z-index:1}
.hero-l{display:flex;gap:18px;align-items:center}
.logo{width:56px;height:56px;border-radius:16px;display:grid;place-items:center;background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.28);box-shadow:0 8px 24px -8px rgba(0,0,0,.5)}
.eyebrow{font-size:11px;letter-spacing:.2em;color:#c7d2fe;font-weight:600}
.hero h1{margin:3px 0 5px;font-size:32px;line-height:1.1;font-weight:700;letter-spacing:-.025em;color:#fff}
.hero p{margin:0;color:#e0e7ff;opacity:.92;font-size:14px;max-width:660px;line-height:1.5}
.hero-r{display:flex;gap:8px;flex-wrap:wrap;max-width:360px;justify-content:flex-end}
.chip{font-size:12px;padding:6px 13px;border-radius:999px;background:rgba(255,255,255,.10);border:1px solid rgba(255,255,255,.22);color:#eef2ff;backdrop-filter:blur(6px)}
.notice{margin:12px 4px 14px;font-size:12.5px;color:var(--mut);display:flex;gap:9px;align-items:center}
.notice::before{content:"";flex:none;width:7px;height:7px;border-radius:50%;background:#f59e0b;box-shadow:0 0 10px #f59e0b}
.toolbar{background:var(--surface);border:1px solid var(--line);border-radius:18px;padding:12px 16px !important;align-items:center !important;gap:16px !important}
.toolbar .block,.toolbar .form{border:none !important;background:transparent !important;box-shadow:none !important}
.who{display:flex;align-items:center;gap:12px}
.avatar{width:40px;height:40px;border-radius:50%;display:grid;place-items:center;font-weight:700;font-size:16px;color:#fff;background:linear-gradient(135deg,#6366f1,#22d3ee)}
.who b{display:block;font-size:14px;color:var(--tx);font-weight:600}.who span{font-size:12px;color:var(--mut)}
.pill{display:inline-flex;align-items:center;gap:9px;padding:8px 15px;border-radius:999px;font-size:13px;border:1px solid var(--line2);background:var(--surface2);color:var(--tx)}
.dot{width:8px;height:8px;border-radius:50%}
.dot.on{background:#22c55e;box-shadow:0 0 10px #22c55e}.dot.off{background:#64748b}.dot.err{background:#ef4444;box-shadow:0 0 10px #ef4444}
.tab-nav button,button[role="tab"]{font-size:14px !important;font-weight:500 !important;color:var(--mut) !important;padding:11px 20px !important;background:transparent !important;border:none !important;border-bottom:2px solid transparent !important;transition:color .15s}
.tab-nav button:hover,button[role="tab"]:hover{color:var(--tx) !important}
.tab-nav button.selected,button[role="tab"][aria-selected="true"]{color:#c7d2fe !important;border-bottom:2px solid var(--acc) !important}
.tab-nav,div[role="tablist"]{border-bottom:1px solid var(--line) !important}
.sec{margin:6px 0 10px}.sec-t{font-size:18px;font-weight:650;letter-spacing:-.01em;color:var(--tx)}.sec-s{font-size:13px;color:var(--mut);margin-top:3px;line-height:1.5}
.panel{background:var(--surface) !important;border:1px solid var(--line) !important;border-radius:18px !important;padding:18px !important}
.kpi-wrap{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:14px;margin:6px 0 12px}
.kpi{position:relative;overflow:hidden;background:linear-gradient(180deg,var(--surface2),var(--surface));border:1px solid var(--line);border-radius:16px;padding:18px 20px 15px}
.kpi::before{content:"";position:absolute;left:0;right:0;top:0;height:3px;background:var(--c,#6366f1)}
.kpi::after{content:"";position:absolute;right:-34px;top:-34px;width:120px;height:120px;border-radius:50%;background:var(--c,#6366f1);opacity:.14;filter:blur(20px)}
.kpi .v{font-size:34px;font-weight:700;color:#f8fafc;letter-spacing:-.025em;line-height:1.05}
.kpi .l{margin-top:8px;font-size:11.5px;color:var(--mut);text-transform:uppercase;letter-spacing:.09em;font-weight:600}
.badge{display:inline-block;padding:3px 11px;border-radius:999px;font-size:11.5px;font-weight:600;margin:0 6px 5px 0;color:var(--c);
 background:color-mix(in srgb,var(--c) 16%,transparent);border:1px solid color-mix(in srgb,var(--c) 42%,transparent)}
.tcard{background:linear-gradient(180deg,var(--surface2),var(--surface));color:var(--tx);border:1px solid var(--line);border-left:4px solid var(--c,#6366f1);border-radius:14px;padding:16px 20px;margin:12px 0}
.tcard h4{margin:0 0 10px;font-size:15px;font-weight:600;color:#f1f5f9;line-height:1.4}
.tcard .id{font-family:'JetBrains Mono',ui-monospace,monospace;color:#a5b4fc;font-size:12.5px;margin-right:10px}
.tcard .meta{font-size:13px;color:var(--mut);line-height:1.6;margin:8px 0 0}.tcard .meta b{color:var(--tx);font-weight:600}
.tcard ul{margin:10px 0 0 18px;padding:0;color:#cbd5e1;font-size:13.5px;line-height:1.55}.tcard li{margin:3px 0}
.tcard .src{margin-top:12px;padding-top:10px;border-top:1px dashed var(--line2);font-size:11.5px;color:#64748b;font-family:'JetBrains Mono',ui-monospace,monospace}
.tcard.err{--c:#ef4444}
.empty{border:1px dashed var(--line2);border-radius:18px;padding:48px 20px;text-align:center;color:var(--mut);font-size:13.5px}
.empty .ic{font-size:30px;opacity:.85}.empty b{display:block;color:var(--tx);margin:10px 0 4px;font-size:15px;font-weight:600}
button.primary{border:none !important;font-weight:600 !important;box-shadow:0 10px 24px -12px rgba(99,102,241,.9);transition:transform .12s,box-shadow .12s}
button.primary:hover{transform:translateY(-1px);box-shadow:0 14px 28px -12px rgba(99,102,241,1)}
button.secondary{font-weight:500 !important}
.gradio-container table thead th{background:#0a1226 !important;color:var(--mut) !important;text-transform:uppercase;font-size:11.5px !important;letter-spacing:.06em}
.gradio-container table td{font-size:13px !important}
.gradio-container .progress-text,.gradio-container .meta-text{font-family:'JetBrains Mono',monospace !important;font-size:12px !important;color:var(--mut) !important}
.foot{margin:28px 0 10px;text-align:center;font-size:12px;color:#64748b}
"""

SHIELD = ("<svg width='28' height='28' viewBox='0 0 24 24' fill='none' stroke='white' stroke-width='1.7' stroke-linecap='round' stroke-linejoin='round'>"
          "<path d='M12 3l8 3v6c0 4.5-3.2 8-8 9-4.8-1-8-4.5-8-9V6l8-3z'/><path d='M8.5 12.5l2.5 2.5 4.5-5'/></svg>")
HERO = (f"<div class='hero'><div class='hero-l'><div class='logo'>{SHIELD}</div><div>"
        "<div class='eyebrow'>AUTOMOTIVE CYBERSECURITY</div><h1>TARA Assistant</h1>"
        "<p>Threat analysis and risk assessment grounded in your own knowledge base. Every finding is cited, scored and auditable.</p></div></div>"
        "<div class='hero-r'><span class='chip'>RAG-grounded</span><span class='chip'>Cited sources</span>"
        "<span class='chip'>Confidence scoring</span><span class='chip'>Versioned</span><span class='chip'>Hash-chained audit</span></div></div>"
        "<div class='notice'>Human approval required. This assistant supports engineering decision-making; it does not replace it.</div>")

def section(title, sub=""): return f"<div class='sec'><div class='sec-t'>{title}</div><div class='sec-s'>{sub}</div></div>"
def empty_state(icon, title, sub): return f"<div class='empty'><div class='ic'>{icon}</div><b>{title}</b>{sub}</div>"
def who_html(k):
    u = USERS[k]; n = len(u["projects"])
    return (f"<div class='who'><div class='avatar'>{u['user'][0].upper()}</div><div><b>{u['user'].title()}</b>"
            f"<span>{u['role'].replace('_', ' ').title()} &middot; {n} project{'s' if n != 1 else ''}</span></div></div>")
def llm_pill(msg):
    m = str(msg); st = "on" if m.startswith("LLM narratives ON") else ("err" if "could not" in m else "off")
    label = "Fast template mode" if st == "off" else html.escape(m[:90])
    return f"<span class='pill'><span class='dot {st}'></span>{label}</span>"

def kpi(items):
    return "<div class='kpi-wrap'>" + "".join(
        f"<div class='kpi' style='--c:{c}'><div class='v'>{v}</div><div class='l'>{l}</div></div>" for l, v, c in items) + "</div>"
def badge(text, color): return f"<span class='badge' style='--c:{color}'>{html.escape(str(text))}</span>"
def empty_fig(msg):
    fig, ax = plt.subplots(figsize=(8, 2.5)); fig.patch.set_facecolor("#0d1424"); ax.set_facecolor("#0d1424"); ax.axis("off")
    ax.text(.5, .5, msg, ha="center", va="center", fontsize=13, color="#64748b"); plt.close(fig); return fig
def err_card(r):
    try: d = r.json().get("detail")
    except Exception: d = r.text[:200]
    return f"<div class='tcard err'>Error {r.status_code}: {html.escape(str(d))}</div>"

def latest_records(key):
    u = USERS[key]
    return [v[-1] for v in RECORDS.values() if v[-1]["project"] in u["projects"]]
def my_ids(key): return [r["tara_id"] for r in latest_records(key)]

def set_llm(on):
    global llm
    if not on: llm = None; return "LLM narratives OFF (fast template mode)"
    if llm_backup[0] is None:
        try:
            from transformers import pipeline
            llm_backup[0] = pipeline("text-generation", model=LLM_NAME, torch_dtype="auto", device_map="auto")
        except Exception as e:
            llm = None; return f"LLM could not load: {e}"
    llm = llm_backup[0]; return f"LLM narratives ON ({LLM_NAME})"

# ---------------- Dashboard ----------------
def ui_dashboard(key):
    recs = latest_records(key)
    rows = [r for rec in recs for r in rec["rows"]]
    kp = kpi([("TARAs", len(recs), "#6366f1"),
              ("Approved", sum(r["status"] == "APPROVED" for r in recs), "#16a34a"),
              ("Threat scenarios", len(rows), "#0ea5e9"),
              ("High / Critical risks", sum(r["risk_value"] >= 4 for r in rows), "#ef4444"),
              ("Need expert review", sum(r["needs_expert_review"] for r in rows), "#d97706")])
    if not rows: return kp, empty_fig("No TARAs yet. Create one in the 'New TARA' tab."), pd.DataFrame()
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
    cnt = [sum(r["risk_value"] == i for r in rows) for i in range(1, 6)]
    axs[0].bar([RISK_LABEL[i] for i in range(1, 6)], cnt, color=[PALETTE[i] for i in range(1, 6)]); axs[0].set_title("Risk distribution")
    axs[0].tick_params(axis="x", rotation=25)
    st = pd.Series([r["stride"] for r in rows]).value_counts()
    axs[1].barh(st.index[::-1], st.values[::-1], color="#6366f1"); axs[1].set_title("Threats by STRIDE category")
    cf = [sum(r["confidence"] == c for r in rows) for c in ["High", "Medium", "Low"]]
    nz = [(l, v, CONF_COL[l]) for l, v in zip(["High", "Medium", "Low"], cf) if v]
    axs[2].pie([v for _, v, _ in nz], labels=[l for l, _, _ in nz], colors=[c for _, _, c in nz], autopct=lambda p: f"{p:.0f}%", wedgeprops=dict(width=.45, edgecolor="#0d1424"))
    axs[2].set_title("Retrieval confidence")
    fig.tight_layout(); plt.close(fig)
    df = pd.DataFrame([dict(TARA=r["tara_id"], Project=r["project"], Asset=r["asset"]["name"], Version=r["version"], Status=r["status"],
                            Threats=len(r["rows"]), MaxRisk=max([x["risk_value"] for x in r["rows"]] or [0]), By=r["created_by"]) for r in recs])
    return kp, fig, df

# ---------------- New TARA ----------------
def cards_html(rec):
    ok = rec["status"] == "APPROVED"
    head = (f"<div class='tcard' style='--c:#6366f1'><h4><span class='id'>{rec['tara_id']}</span>Version {rec['version']}</h4>"
            f"{badge(rec['status'], '#22c55e' if ok else '#f59e0b')}"
            f"<p class='meta'>Asset: <b>{html.escape(rec['asset']['name'])}</b> &nbsp;|&nbsp; Project: <b>{rec['project']}</b> &nbsp;|&nbsp; By: <b>{rec['created_by']}</b></p></div>")
    if rec.get("warning"): head += f"<div class='tcard' style='--c:#f59e0b'>{html.escape(rec['warning'])}</div>"
    for r in rec["rows"]:
        c = r["citations"][0]
        head += (f"<div class='tcard' style='--c:{PALETTE[r['risk_value']]}'>"
                 f"<h4><span class='id'>{r['threat_id']}</span>{html.escape(r['threat_scenario'][:140])}</h4>"
                 f"{badge('Risk ' + str(r['risk_value']) + ' - ' + RISK_LABEL[r['risk_value']], PALETTE[r['risk_value']])}"
                 f"{badge(r['stride'], '#818cf8')}{badge('Impact ' + r['impact_rating'], '#94a3b8')}{badge('Feasibility ' + r['feasibility'], '#38bdf8')}"
                 f"{badge('Confidence ' + r['confidence'], CONF_COL[r['confidence']])}"
                 f"{badge('Expert review', '#ef4444') if r['needs_expert_review'] else ''}"
                 f"<p class='meta'>Damage: {html.escape(r['damage_scenario'])}<br>Treatment: <b>{r['treatment']}</b> &middot; Attack potential sum: <b>{r['attack_potential_sum']}</b></p>"
                 f"<ul>{''.join('<li>' + html.escape(x) + '</li>' for x in r['controls'])}</ul>"
                 f"<div class='src'>{html.escape(c['title'])} ({c['doc_id']}) &middot; {html.escape(c['source'])} &middot; similarity {c['similarity']}</div></div>")
    return head

def tara_fig(rec):
    if not rec["rows"]: return empty_fig("No grounded threats retrieved")
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))
    ax = axs[0]
    grid = np.array([MATRIX[i] for i in range(4)])
    ax.grid(False)
    ax.imshow(grid, cmap=ListedColormap([PALETTE[i] for i in range(1, 6)]), vmin=1, vmax=5, origin="lower", alpha=.55)
    for i in range(4):
        for j in range(4): ax.text(j, i - .38, RISK_LABEL[grid[i, j]], ha="center", va="center", fontsize=7, color="#e2e8f0")
    seen = {}
    for r in rec["rows"]:
        x, y = FEAS_IDX[r["feasibility"]], IMPACT_NAMES.index(r["impact_rating"])
        n = seen.get((x, y), 0); seen[(x, y)] = n + 1
        ax.scatter(x - .25 + .25 * n, y, s=170, color="#f8fafc", edgecolors="#0d1424", zorder=3)
        ax.text(x - .25 + .25 * n, y + .17, r["threat_id"].split("-")[-1], ha="center", fontsize=7, zorder=4)
    ax.set_xticks(range(4)); ax.set_xticklabels(["Very Low", "Low", "Medium", "High"]); ax.set_yticks(range(4)); ax.set_yticklabels(IMPACT_NAMES)
    ax.set_xlabel("Attack feasibility"); ax.set_ylabel("Impact"); ax.set_title("Risk heat map (dots = threat numbers)")
    ax = axs[1]
    rows = rec["rows"][::-1]
    ax.barh([r["threat_id"] for r in rows], [r["risk_value"] for r in rows], color=[PALETTE[r["risk_value"]] for r in rows])
    for i, r in enumerate(rows): ax.text(r["risk_value"] + .05, i, f"{r['feasibility']} | conf {r['confidence']}", va="center", fontsize=8)
    ax.set_xlim(0, 7); ax.set_xlabel("Risk value (1-5)"); ax.set_title("Risk per threat")
    fig.tight_layout(); plt.close(fig); return fig

def ui_generate(key, project, name, atype, desc, ifaces, use_llm):
    if (llm is not None) != bool(use_llm): set_llm(use_llm)      # only touch the model when the switch changed
    r = requests.post(f"{API}/projects/{project}/tara", headers=H(key), json={"name": name, "asset_type": atype, "description": desc, "interfaces": ifaces})
    if r.status_code != 200:
        return err_card(r), empty_fig("Not allowed"), pd.DataFrame(), gr.update()
    rec = r.json()
    return cards_html(rec), tara_fig(rec), rows_to_df(rec), gr.update(choices=my_ids(key), value=rec["tara_id"])

# ---------------- Review and approval ----------------
NO_TARA = empty_state("&#128269;", "No TARA selected", "Pick a TARA ID above and press Load.")
def ui_view(key, tid):
    if not tid: return NO_TARA, pd.DataFrame(), empty_fig("Select a TARA"), pd.DataFrame()
    r = requests.get(f"{API}/tara/{tid}", headers=H(key))
    if r.status_code != 200: return err_card(r), pd.DataFrame(), empty_fig("No access"), pd.DataFrame()
    rec = r.json(); vers = RECORDS[tid]
    vdf = pd.DataFrame([dict(Version=v["version"], Status=v["status"], ApprovedBy=v["approved_by"]) for v in vers])
    if not vers[0]["rows"]: return cards_html(rec), rows_to_df(rec), empty_fig("No rows"), vdf
    fig, ax = plt.subplots(figsize=(10, 3.8))
    n = len(vers); w = .8 / n
    for vi, v in enumerate(vers):
        ax.bar(np.arange(len(v["rows"])) + vi * w, [x["risk_value"] for x in v["rows"]], w, label=f"v{v['version']}")
    ax.set_xticks(np.arange(len(vers[0]["rows"])) + .4 - w / 2); ax.set_xticklabels([f"#{i} {x['threat_id'].split('-')[-1]}" for i, x in enumerate(vers[0]["rows"])], rotation=20)
    ax.set_ylabel("Risk value"); ax.set_title("Risk by row across versions"); ax.legend()
    fig.tight_layout(); plt.close(fig)
    return cards_html(rec), rows_to_df(rec), fig, vdf

def _msg(r, ok):
    if r.status_code == 200: return ok(r.json())
    try: return f"Error {r.status_code}: {r.json().get('detail')}"
    except Exception: return f"Error {r.status_code}"
def ui_revise(key, tid, idx, et, ex, kn, wo, eq, note):
    if not tid: return "Select a TARA first."
    r = requests.post(f"{API}/tara/{tid}/revise", headers=H(key),
                      json=dict(row_index=int(idx or 0), attack_potential=[int(et or 0), int(ex or 0), int(kn or 0), int(wo or 0), int(eq or 0)], note=note or "manual revision"))
    return _msg(r, lambda j: f"Revised: now v{j['version']}")
def ui_approve(key, tid):
    if not tid: return "Select a TARA first."
    return _msg(requests.post(f"{API}/tara/{tid}/approve", headers=H(key)), lambda j: f"Approved by {j['approved_by']}")

# ---------------- Accuracy and evaluation (searches are cached, sweeps are instant) ----------------
EVAL_SET = [
 ("project_adas", "Gateway ECU", "ECU", "Central gateway routing CAN and Ethernet traffic between vehicle domains", "CAN, Ethernet", {"THR-ADAS-001", "THR-ADAS-007"}),
 ("project_adas", "OTA update client", "OTA", "Telematics unit that downloads and installs signed firmware over cellular", "cellular, telematics", {"THR-ADAS-002", "THR-ADAS-006"}),
 ("project_adas", "Perception module", "sensor", "Camera radar and GNSS sensors feeding ADAS fusion", "GNSS, camera, radar", {"THR-ADAS-004"}),
 ("project_adas", "Diagnostic port", "diagnostic", "OBD-II connector supporting UDS security access and reflashing", "OBD-II, UDS", {"THR-ADAS-003"}),
 ("project_adas", "Bootloader and HSM", "bootloader", "Secure boot and key storage with debug interfaces", "JTAG, SWD", {"THR-ADAS-005", "THR-ADAS-008"}),
 ("project_infotainment", "Head unit media", "IVI", "Infotainment head unit with USB media and Bluetooth phone connection", "USB, Bluetooth", {"THR-IVI-001", "THR-IVI-002"}),
 ("project_infotainment", "App platform", "app store", "Third-party apps installed on the infotainment system", "app store", {"THR-IVI-003"}),
]
# Illustrative expert reference ratings. Replace with risk values from your approved past TARAs.
EXPERT_RISK = {"THR-ADAS-001": 5, "THR-ADAS-002": 3, "THR-ADAS-003": 3, "THR-ADAS-004": 3, "THR-ADAS-005": 2, "THR-ADAS-006": 4,
               "THR-ADAS-007": 3, "THR-ADAS-008": 2, "THR-IVI-001": 3, "THR-IVI-002": 3, "THR-IVI-003": 3}

_ranked = {}
def ranked(it):
    key_ = (it[0], it[1], len(store.meta[it[0]]))     # re-ranks automatically if the KB grows
    if key_ not in _ranked:
        proj, name, atype, desc, ifc, _ = it
        _ranked[key_] = [(m, s) for m, s in store.search(proj, f"{atype} {name} {desc} {ifc}", k=8) if m["type"] == "threat"]
    return _ranked[key_]
def predict(it, thr, k): return [(m, s) for m, s in ranked(it) if s >= thr][:k]
def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)
def metrics(thr, k):
    per, TP, FP, FN = [], 0, 0, 0
    for it in EVAL_SET:
        ids = {m["id"] for m, _ in predict(it, thr, k)}; exp = it[5]
        tp, fp, fn = len(ids & exp), len(ids - exp), len(exp - ids); TP += tp; FP += fp; FN += fn
        per.append((it[1],) + prf(tp, fp, fn))
    return per, prf(TP, FP, FN)
def engine_risk(meta): return risk_value(max(meta["impact"].values()), feasibility(meta["ap"])[1])

def ui_eval(thr, k):
    k = int(k); per, (P, R, F) = metrics(thr, k)
    allmeta = {m["id"]: m for c in store.meta.values() for m in c if m["type"] == "threat"}
    pairs = [(EXPERT_RISK[i], engine_risk(allmeta[i]), i) for i in EXPERT_RISK]
    exact = np.mean([a == b for a, b, _ in pairs]); within1 = np.mean([abs(a - b) <= 1 for a, b, _ in pairs]); mae = np.mean([abs(a - b) for a, b, _ in pairs])
    kp = kpi([("Precision", f"{P:.0%}", "#6366f1"), ("Recall", f"{R:.0%}", "#0ea5e9"), ("F1 score", f"{F:.0%}", "#8b5cf6"),
              ("Risk exact match", f"{exact:.0%}", "#16a34a"), ("Risk within +/-1", f"{within1:.0%}", "#22c55e"), ("Risk MAE", f"{mae:.2f}", "#d97706")])
    fig, axs = plt.subplots(2, 2, figsize=(14, 9))
    ax = axs[0, 0]; x = np.arange(len(per))
    for i, (lab, col) in enumerate([("Precision", "#6366f1"), ("Recall", "#0ea5e9"), ("F1", "#8b5cf6")]):
        ax.bar(x + (i - 1) * .27, [p[i + 1] for p in per], .27, label=lab, color=col)
    ax.set_xticks(x); ax.set_xticklabels([p[0] for p in per], rotation=25, ha="right"); ax.set_ylim(0, 1.05); ax.legend(); ax.set_title("Threat retrieval accuracy per test asset")
    ax = axs[0, 1]; ths = np.arange(.15, .66, .05); sw = [metrics(t, k)[1] for t in ths]
    for i, (lab, col) in enumerate([("Precision", "#6366f1"), ("Recall", "#0ea5e9"), ("F1", "#8b5cf6")]):
        ax.plot(ths, [s[i] for s in sw], marker="o", label=lab, color=col)
    ax.axvline(thr, color="#ef4444", ls="--", label="chosen threshold"); ax.set_xlabel("Similarity threshold"); ax.set_ylim(0, 1.05); ax.legend(); ax.set_title("Precision / recall trade-off")
    ax = axs[1, 0]; rng = np.random.default_rng(1)
    ex = np.array([p[0] for p in pairs]) + rng.uniform(-.12, .12, len(pairs)); en = np.array([p[1] for p in pairs]) + rng.uniform(-.12, .12, len(pairs))
    ax.fill_between([.5, 5.5], [-.5, 4.5], [1.5, 6.5], color="#22c55e", alpha=.14, label="within +/-1"); ax.plot([.5, 5.5], [.5, 5.5], "--", color="#94a3b8", lw=1)
    ax.scatter(ex, en, s=70, color="#6366f1", zorder=3)
    for (a, b, i) in pairs: ax.annotate(i.split("-")[-1], (a + .05, b + .05), fontsize=7)
    ax.set_xlim(.5, 5.5); ax.set_ylim(.5, 5.5); ax.set_xlabel("Expert risk"); ax.set_ylabel("Assistant risk"); ax.legend(loc="upper left"); ax.set_title("Risk rating agreement")
    ax = axs[1, 1]; good, bad = [], []
    for it in EVAL_SET:
        for m, s in predict(it, 0.0, 8): (good if m["id"] in it[5] else bad).append(s)
    ax.hist([good, bad], bins=12, label=["Correct threat", "Wrong threat"], color=["#16a34a", "#dc2626"], stacked=True)
    ax.axvline(.35, color="#d97706", ls="--"); ax.axvline(.5, color="#16a34a", ls="--"); ax.set_xlabel("Similarity score"); ax.set_title("Do confidence levels separate right from wrong?"); ax.legend()
    fig.tight_layout(); plt.close(fig)
    tbl = pd.DataFrame([dict(Asset=p[0], Precision=round(p[1], 2), Recall=round(p[2], 2), F1=round(p[3], 2)) for p in per])
    return kp, fig, tbl

# ---------------- Audit ----------------
def ui_audit(key):
    r = requests.get(f"{API}/audit", headers=H(key))
    u = USERS[key]
    if r.status_code == 403:
        return (empty_state("&#128274;", "Audit log is restricted",
                f"{u['user'].title()} ({u['role'].replace('_', ' ')}) has no audit access. Switch to a security lead or auditor key to view the log."),
                pd.DataFrame(), None)
    if r.status_code != 200: return err_card(r), pd.DataFrame(), None
    d = r.json(); df = pd.DataFrame(d["entries"])
    ok = badge("Hash chain valid" if d["chain_valid"] else "CHAIN BROKEN", "#16a34a" if d["chain_valid"] else "#dc2626")
    if df.empty: return ok + " no entries yet", df, None
    df["details"] = df["details"].astype(str)
    fig, ax = plt.subplots(figsize=(8, 3)); vc = df["action"].value_counts(); ax.bar(vc.index, vc.values, color="#6366f1"); ax.set_title("Audit events by action")
    fig.tight_layout(); plt.close(fig)
    return ok + f" {len(df)} events", df[["ts", "user", "action", "project", "details"]], fig

# ---------------- Knowledge base ----------------
def ui_kb(key, project):
    try: authorize(USERS[key], project, "read")
    except PermissionError as e: return pd.DataFrame([dict(error=str(e))])
    return pd.DataFrame([dict(ID=m["id"], Type=m["type"], Title=m["title"], STRIDE=m.get("stride", ""), Source=m["source"]) for m in store.meta.get(project, [])])
def ui_ingest(key, project, title, text, source):
    r = requests.post(f"{API}/projects/{project}/ingest", headers=H(key), json=dict(title=title, text=text, source=source or "User upload"))
    return _msg(r, lambda j: f"Indexed {j['chunks']} chunk(s)"), ui_kb(key, project)

# ---------------- Session change: refresh EVERY tab so nothing shows another user's stale result ----------------
def on_user(k, proj):
    kp, fig, df = ui_dashboard(k)
    ab, at, af = ui_audit(k)
    return (who_html(k), kp, fig, df, ab, at, af,
            gr.update(choices=my_ids(k), value=None), NO_TARA, ui_kb(k, proj))

# ---------------- Layout ----------------
RISK_NOTE = "Attack potential points: elapsed time, expertise, knowledge, window of opportunity, equipment."
EMPTY_RESULTS = empty_state("&#128737;", "No draft yet", "Describe an asset on the left and press <b style='display:inline;font-size:inherit'>Generate TARA draft</b>.")
_ui = dict(theme=THEME, css=CSS, js=FORCE_DARK)
_in_launch = "theme" in _inspect.signature(gr.Blocks.launch).parameters   # Gradio 6+ moved theme/css/js to launch()

with gr.Blocks(title="TARA Assistant", **({} if _in_launch else _ui)) as demo:
    gr.HTML(HERO)
    with gr.Row(elem_classes="toolbar", equal_height=True):
        with gr.Column(scale=3, min_width=240):
            key = gr.Dropdown(list(USERS), value="analyst-key", label="Signed in as", info="API key (demo authentication)")
        with gr.Column(scale=2, min_width=200):
            who = gr.HTML(who_html("analyst-key"))
        with gr.Column(scale=2, min_width=200):
            use_llm = gr.Checkbox(False, label="LLM narratives", info="Slower, richer text")
        with gr.Column(scale=3, min_width=240):
            llm_msg = gr.HTML(llm_pill("LLM narratives OFF (fast template mode)"))
    use_llm.change(lambda on: llm_pill(set_llm(on)), use_llm, llm_msg, show_progress="minimal")

    with gr.Tabs():
        with gr.Tab("Dashboard"):
            with gr.Row():
                gr.HTML(section("Portfolio overview", "Risk posture across your latest TARA versions"))
                d_btn = gr.Button("Refresh", variant="secondary", scale=0, min_width=130)
            d_kpi = gr.HTML(); d_fig = gr.Plot(show_label=False)
            d_tbl = gr.Dataframe(label="TARA records (latest versions)", interactive=False, wrap=True)
            d_btn.click(ui_dashboard, key, [d_kpi, d_fig, d_tbl], show_progress="minimal")

        with gr.Tab("New TARA"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=2, min_width=340, elem_classes="panel"):
                    gr.HTML(section("Asset definition", "Threats are retrieved from the selected project collection"))
                    project = gr.Dropdown(PROJECTS, value="project_adas", label="Project collection")
                    with gr.Row():
                        name = gr.Textbox("ADAS Domain Controller", label="Asset name"); atype = gr.Textbox("ECU", label="Asset type")
                    desc = gr.Textbox("Fuses camera and radar, sends brake requests on CAN, supports UDS reflashing and OTA updates", label="Description", lines=3)
                    ifaces = gr.Textbox("CAN, Ethernet, OBD-II, telematics", label="Interfaces")
                    g_btn = gr.Button("Generate TARA draft", variant="primary", size="lg")
                with gr.Column(scale=5, min_width=420):
                    gr.HTML(section("Results", "Heat map, per-threat risk, full table and cited threat cards"))
                    with gr.Tabs():
                        with gr.Tab("Risk overview"): g_fig = gr.Plot(show_label=False)
                        with gr.Tab("TARA table"): g_tbl = gr.Dataframe(label="TARA table", interactive=False, wrap=True)
                        with gr.Tab("Threat cards"): g_cards = gr.HTML(EMPTY_RESULTS)

        with gr.Tab("Review and Approval") as t_review:
            gr.HTML(section("Review and approval", "Inspect a draft, record human revisions, and approve as security lead"))
            with gr.Row(equal_height=True):
                review_dd = gr.Dropdown([], label="TARA ID", scale=4)
                r_refresh = gr.Button("Refresh list", variant="secondary", scale=1, min_width=130)
                r_load = gr.Button("Load", variant="primary", scale=1, min_width=130)
            r_cards = gr.HTML(NO_TARA); r_fig = gr.Plot(show_label=False)
            with gr.Row():
                r_tbl = gr.Dataframe(label="Current rows", interactive=False, wrap=True); r_ver = gr.Dataframe(label="Version history", interactive=False)
            with gr.Accordion("Human revision (creates a new version and resets approval)", open=True):
                gr.HTML(f"<div class='sec-s' style='margin-bottom:8px'>{RISK_NOTE}</div>")
                with gr.Row():
                    ridx = gr.Number(0, label="Row index", precision=0); et = gr.Number(4, label="Elapsed time", precision=0); ex = gr.Number(6, label="Expertise", precision=0)
                    kn = gr.Number(3, label="Knowledge", precision=0); wo = gr.Number(4, label="Window of opportunity", precision=0); eq = gr.Number(4, label="Equipment", precision=0)
                rnote = gr.Textbox(label="Justification", placeholder="Why is this rating being changed?")
                with gr.Row():
                    rev_btn = gr.Button("Save revision", variant="secondary"); app_btn = gr.Button("Approve (security_lead only)", variant="primary")
                r_msg = gr.Textbox(label="Result", interactive=False)

        with gr.Tab("Accuracy and Evaluation"):
            gr.HTML(section("Accuracy and evaluation", "Benchmarks against a labelled test set (illustrative; replace with your approved past TARAs). Reports retrieval precision, recall and F1, plus agreement of risk ratings with expert ratings."))
            with gr.Row(equal_height=True):
                thr = gr.Slider(.15, .6, value=.25, step=.05, label="Similarity threshold"); topk = gr.Slider(1, 8, value=4, step=1, label="Top-k threats")
                e_btn = gr.Button("Run evaluation", variant="primary", scale=0, min_width=180)
            e_kpi = gr.HTML(); e_fig = gr.Plot(show_label=False); e_tbl = gr.Dataframe(label="Per-asset results", interactive=False)
            e_btn.click(ui_eval, [thr, topk], [e_kpi, e_fig, e_tbl], show_progress="minimal")

        with gr.Tab("Audit Log") as t_audit:
            with gr.Row():
                gr.HTML(section("Audit trail", "Tamper-evident log of every generate, revise, approve and ingest action. Reloads automatically when you open this tab or switch user."))
                a_btn = gr.Button("Reload audit log", variant="primary", scale=0, min_width=170)
            a_badge = gr.HTML(); a_fig = gr.Plot(show_label=False); a_tbl = gr.Dataframe(interactive=False, wrap=True)
            a_btn.click(ui_audit, key, [a_badge, a_tbl, a_fig], show_progress="minimal")
            t_audit.select(ui_audit, key, [a_badge, a_tbl, a_fig], show_progress="hidden")

        with gr.Tab("Knowledge Base"):
            gr.HTML(section("Knowledge base", "Documents the assistant retrieves from. Add your own threat catalogs and past TARAs."))
            with gr.Row(equal_height=True):
                kb_proj = gr.Dropdown(PROJECTS, value="project_adas", label="Project collection", scale=4)
                kb_btn = gr.Button("List documents", variant="secondary", scale=1, min_width=160)
            kb_tbl = gr.Dataframe(interactive=False, wrap=True)
            with gr.Accordion("Add a document", open=False):
                with gr.Row():
                    kb_title = gr.Textbox(label="Document title"); kb_src = gr.Textbox(label="Source reference")
                kb_text = gr.Textbox(lines=5, label="Text (paragraphs become chunks)")
                kb_ing = gr.Button("Ingest into project collection", variant="primary"); kb_msg = gr.Textbox(label="Result", interactive=False)
            kb_btn.click(ui_kb, [key, kb_proj], kb_tbl, show_progress="minimal")
            kb_proj.change(ui_kb, [key, kb_proj], kb_tbl, show_progress="minimal")
            kb_ing.click(ui_ingest, [key, kb_proj, kb_title, kb_text, kb_src], [kb_msg, kb_tbl])

    gr.HTML("<div class='foot'>TARA Assistant &middot; pilot build &middot; sample knowledge base is illustrative &middot; all outputs require human review</div>")

    # every tab is refreshed whenever the signed-in user changes (fixes the stale "403" bug)
    _sync_out = [who, d_kpi, d_fig, d_tbl, a_badge, a_tbl, a_fig, review_dd, r_cards, kb_tbl]
    key.change(on_user, [key, kb_proj], _sync_out, show_progress="hidden")
    demo.load(on_user, [key, kb_proj], _sync_out)
    t_review.select(lambda k: gr.update(choices=my_ids(k)), key, review_dd, show_progress="hidden")

    g_btn.click(ui_generate, [key, project, name, atype, desc, ifaces, use_llm], [g_cards, g_fig, g_tbl, review_dd])
    r_refresh.click(lambda k: gr.update(choices=my_ids(k)), key, review_dd, show_progress="hidden")
    r_load.click(ui_view, [key, review_dd], [r_cards, r_tbl, r_fig, r_ver], show_progress="minimal")
    rev_btn.click(ui_revise, [key, review_dd, ridx, et, ex, kn, wo, eq, rnote], r_msg).then(ui_view, [key, review_dd], [r_cards, r_tbl, r_fig, r_ver])
    app_btn.click(ui_approve, [key, review_dd], r_msg).then(ui_view, [key, review_dd], [r_cards, r_tbl, r_fig, r_ver])

demo.launch(share=SHARE_LINK, debug=False, **(_ui if _in_launch else {}))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c81f48969ab9b6538.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
